# 05 · v1 vs v2, measured
- 20 questions with known answers: causes, times, versions, IPs, fixes, and 3 **trap** questions about things that never happened
- An answer counts as correct if it contains an expected word (for traps: says it is not there)
- Also measured: did v2's search bring back the right evidence?

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'          # models come from the local cache only
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.max_colwidth', 120)
from logbot.parse import read_logs
from logbot.llm import LocalLLM
from logbot.chat import ChatV1
from logbot.rag import RagIndex, AskV2
from logbot.evaluate import QUESTIONS, run
df = read_logs(); llm = LocalLLM(); index = RagIndex.load()
v1, v2 = ChatV1(llm, df), AskV2(llm, index, df)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
r1 = run('v1 chatbot', lambda q: (v1.ask(q, session_id=q)[0], None))
r2 = run('v2 RAG', lambda q: v2.ask(q)[:2])
both = pd.concat([r1, r2])
both.groupby('version').agg(correct=('correct', 'mean'), seconds=('seconds', 'mean')).round(2)

,correct,seconds
version,,
v1 chatbot,0.30,0.60
v2 RAG,0.85,0.43


## Question by question

In [3]:
side = r1[['question', 'correct']].rename(columns={'correct': 'v1'}).assign(v2=r2.correct.values)
side

,question,v1,v2
0,What caused the 500 errors on /api/orders on Oct 13?,True,True
1,When did the database connection problem start on Oct 13?,False,True
2,What does the runbook say to do when the Hikari pool is exhausted?,False,True
3,Why did Apache return 502 errors around 11:02 on Oct 14?,False,False
4,What error did Tomcat log before it restarted on Oct 14?,False,True
5,How should we fix a Java heap OutOfMemoryError?,False,True
6,Why was checkout slow on the afternoon of Oct 14?,False,True
7,Which exception did PaymentClient throw?,False,True
8,What should we do when the payment gateway times out?,True,True
9,Which version was deployed on Oct 15 at 14:00?,False,True


## Did search find the right evidence?

In [4]:
import json
from logbot import DATA
key = {k['id']: k for k in json.loads((DATA / 'incidents.json').read_text())}
def evidence_hit(q, inc):
    if inc not in key: return None
    words = key[inc]['keywords']
    return any(any(w in h['text'].lower() for w in words) for h in index.search(q))
r2['evidence_found'] = [evidence_hit(q, inc) for q, _, inc in QUESTIONS]
r2[r2.evidence_found.notna()].evidence_found.mean()

np.float64(1.0)

## Some answers

In [5]:
for i in [0, 3, 12, 17]:
    print('Q:', QUESTIONS[i][0]); print('  v1:', r1.answer[i][:220]); print('  v2:', r2.answer[i][:220]); print()

Q: What caused the 500 errors on /api/orders on Oct 13?
  v1: The 500 error occurred on `/api/orders` on October 13 due to a connection issue with the database.
  v2: The 500 error on `/api/orders` occurred due to a transient connection issue with the database. This was resolved by cancelling the query, moving the reports to the read replica, and setting a 60-second statement timeout 

Q: Why did Apache return 502 errors around 11:02 on Oct 14?
  v1: The error occurred because the connection to the database was closed unexpectedly during the request processing. This could be due to network issues or other external factors that prevented the connection from being esta
  v2: Not found in the logs.

Q: Which IP address attempted the brute-force login?
  v1: The IP address that attempted the brute-force login was 10.0.1.147.
  v2: The IP address that attempted the brute-force login is 203.0.113.77.

Q: Was there a disk full error?
  v1: Yes, there was a disk full error. The log entry shows

In [6]:
acc1, acc2 = r1.correct.mean(), r2.correct.mean()
assert acc2 > acc1, (acc1, acc2)
print(f'✓ v2 (RAG) {acc2:.0%} correct vs v1 {acc1:.0%}; search found the right evidence {r2.evidence_found.dropna().mean():.0%} of the time')

✓ v2 (RAG) 85% correct vs v1 30%; search found the right evidence 100% of the time
